In [ ]:
import bz2
import xml.etree.ElementTree as ET
import mwparserfromhell
import re

## This function extracts plain Georgian-language text from a Wikipedia XML dump file ##
def process_wiki_dump(bz2_file_path, output_txt_path):
    # regex to match only Georgian characters
    georgian_range = re.compile(r'[\u10D0-\u10FA\u10F0-\u10F5]')
    
    print("Processing Wikipedia dump...")
    
    # open the compressed bz2 file and stream it
    with bz2.open(bz2_file_path, 'rt', encoding='utf-8') as bz_file, \
         open(output_txt_path, 'w', encoding='utf-8') as out_file:
        
        # parse the XML context streamingly
        context = ET.iterparse(bz_file, events=('end',))
        
        for event, elem in context:
            # look for the text tag inside a page tracking block
            if elem.tag.endswith('text') and elem.text:
                try:
                    # strip wiki markup (links, templates, tables, infoboxes)
                    wikicode = mwparserfromhell.parse(elem.text)
                    plain_text = wikicode.strip_code()
                    
                    # only write lines that actually contain Georgian characters
                    if georgian_range.search(plain_text):
                        out_file.write(plain_text + '\n')
                except Exception:
                    pass # skip parsing glitches
                
                elem.clear() # free memory from the parsed XML element
                
    print(f"Extraction finished! Raw plain text saved to {output_txt_path}")
    
process_wiki_dump('kawiki-latest-pages-articles.xml.bz2', 'raw_georgian_text.txt')

In [ ]:
from collections import Counter
import re

## This function builds a filtered Georgian word frequency list from the raw text corpus ##
def extract_frequent_georgian_words(input_file_path, output_file_path, min_frequency=4):
    georgian_word_pattern = re.compile(r'^[\u10D0-\u10FA\u10F0-\u10F5]+$')
    word_counter = Counter()
    
    print("Counting word frequencies...")
    with open(input_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            tokens = re.split(r'\s+|[.,!?"\'()\[\]{}::\-_—;=+*@#$%%^&/\\|<>~`]', line)
            for token in tokens:
                token = token.strip()
                if token and georgian_word_pattern.match(token):
                    if 1 < len(token) < 25:
                        word_counter[token] += 1
                        
    # filter out words that don't meet the frequency threshold
    clean_frequent_words = [word for word, count in word_counter.items() if count >= min_frequency]
    
    with open(output_file_path, 'w', encoding='utf-8') as out_f:
        for word in sorted(clean_frequent_words):
            out_f.write(word + '\n')
            
    print(f"Filtering complete!")
    print(f"Original unique words: {len(word_counter)}")
    print(f"Words with frequency >= {min_frequency}: {len(clean_frequent_words)}")

extract_frequent_georgian_words('raw_georgian_text.txt', 'georgian_words.txt', min_frequency=4)

In [ ]:
import random
import pandas as pd
from sklearn.model_selection import train_test_split

## This function synthetically generates typos in Georgian words to create training data for the spellchecker ##
# omit — deletes a random character
# swap — swaps two adjacent characters
# substitute — replaces a character with a random Georgian letter
# none — leaves the word unchanged (so the model also sees clean→clean pairs)

# Examples:
# 1. მოსახლებოა -> მოსახელობა
# 2. გამარჯობა -> გამარჰონა
# 3. საქართველო -> საქარტველო
# 4. ადამიანი -> ადამიანი 
# 5. კომპიუტერი -> კომპიუტერი
def introduce_typo(word):
    if len(word) <= 3:
        return word
    
    word_chars = list(word)
    typo_type = random.choice(['omit', 'swap', 'substitute', 'none']) # 'none' keeps it clean
    
    if typo_type == 'none':
        return word
        
    idx = random.randint(0, len(word_chars) - 1)
    
    if typo_type == 'omit':
        word_chars.pop(idx)
    elif typo_type == 'swap' and idx < len(word_chars) - 1:
        word_chars[idx], word_chars[idx+1] = word_chars[idx+1], word_chars[idx]
    elif typo_type == 'substitute':
        georgian_chars = list("აბგდევზთიკლმნოპჟრსტუფქღყშჩცძწჭხჯჰ")
        word_chars[idx] = random.choice(georgian_chars)
        
    return "".join(word_chars)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

## This function builds the final train/val/test datasets for the seq2seq spellchecker ##
def generate_seq2seq_dataset(dict_path):
    # read all unique clean words
    with open(dict_path, 'r', encoding='utf-8') as f:
        clean_words = [line.strip() for line in f if line.strip()]
    
    # remove duplicates 
    clean_words = list(set(clean_words))
    
    # save a master dataframe containing all target words 
    all_words_df = pd.DataFrame({'word': clean_words})
    all_words_df.to_csv('all_words_dataset.csv', index=False, encoding='utf-8')
    
    # split the core vocabulary first to avoid data leakage
    # 80% train, 10% val, 10% test
    train_words, test_words = train_test_split(clean_words, test_size=0.2, random_state=42)
    val_words, test_words = train_test_split(test_words, test_size=0.5, random_state=42)
    
    def build_pairs(word_list, augment_factor=2):
        source_target_pairs = []
        for word in word_list:
            # always add a perfect pair so the model learns identity mapping
            source_target_pairs.append({'source': word, 'target': word})
            
            # generate multiple distinct typo variations per word
            for _ in range(augment_factor):
                typo_word = introduce_typo(word)
                source_target_pairs.append({'source': typo_word, 'target': word})
                
        return pd.DataFrame(source_target_pairs)

    # create datasets (augmenting train more so the model sees more variations)
    train_df = build_pairs(train_words, augment_factor=3)
    val_df = build_pairs(val_words, augment_factor=1)
    test_df = build_pairs(test_words, augment_factor=1)
    
    # shuffle the datasets
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
    val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)
    test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # save to CSVs
    train_df.to_csv('train_dataset.csv', index=False, encoding='utf-8')
    val_df.to_csv('val_dataset.csv', index=False, encoding='utf-8')
    test_df.to_csv('test_dataset.csv', index=False, encoding='utf-8')
    
    print(f"Dataset Generation Complete!")
    print(f"Master vocabulary file saved to: all_words_dataset.csv")
    print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)} | Test samples: {len(test_df)}")

generate_seq2seq_dataset('georgian_words.txt')